# USLR Linear Registration Tutorial

This notebook walks through the **USLR linear registration** step (`scripts/linear_registration.py`) interactively, using the single subject included in `data/data_tutorial`.

### What this step does
Given a longitudinal T1w MRI dataset (multiple sessions per subject), it:
1. Computes the **center of gravity** of each session's brain segmentation.
2. Runs **pairwise rigid registration** between all session pairs using brain-structure centroids (Kabsch/SVD algorithm).
3. **Solves a global registration graph** (Lie-algebra formulation, LBFGS optimiser) to find consistent per-session transforms to a subject-specific template space.
4. **Builds a subject template** (192³ voxel space) by resampling all sessions and computing the voxel-wise median.
5. Estimates **eTIV** (estimated total intracranial volume) in the template space.
6. Optionally registers the template to **MNI152** space.

### Prerequisite
The linear registration step requires **preprocessed outputs** (SynthSeg segmentations and bias-field-corrected images) produced by the two earlier scripts:
- `scripts/synthseg_segmentation.py`  → `derivatives/preproc/<sub>/<ses>/anat/*T1wdseg.nii.gz`
- `scripts/bias_field_correction.py`  → `derivatives/preproc/<sub>/<ses>/anat/*T1w.nii.gz`

The **Prerequisite check** cell below will tell you whether those outputs are present.

## 1  Environment setup

Must be the **first** executed cell. Sets environment variables consumed by `setup.py` before any package import.

In [ ]:
import os
import sys

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = os.path.abspath('..')           # nicgiprep/
BIDS_DIR  = os.path.join(REPO_ROOT, 'data', 'data_tutorial', 'rawdata')

# ── Environment variables required by setup.py ─────────────────────────────
os.environ['PYTHONPATH']    = REPO_ROOT     # used to locate data/atlas & data/labels_classes_priors
os.environ['BIDS_DIR']      = BIDS_DIR
os.environ['USLR_RUNNING']  = 'True'        # suppresses the interactive FreeSurfer banner in setup.py

# ── sys.path ───────────────────────────────────────────────────────────────
for p in [REPO_ROOT, os.path.join(REPO_ROOT, 'scripts')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'BIDS_DIR  : {BIDS_DIR}')
print(f'BIDS_DIR exists: {os.path.exists(BIDS_DIR)}')

## 2  Imports

In [ ]:
from os.path import exists, join, dirname

import bids
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from setup import *                         # loads DIR_PIPELINES, BIDS_PATH_PATTERN, etc.
from nicgiprep.uslr import USLR_Linear

## 3  Load the BIDS dataset

A SQLite index is cached next to the rawdata root on first load.

In [ ]:
db_file = join(dirname(BIDS_DIR), 'BIDS-raw.db')

if not exists(db_file):
    print('Building BIDS index (first run) ...')
    bids_loader = bids.layout.BIDSLayout(root=BIDS_DIR, validate=False)
    bids_loader.save(db_file)
else:
    bids_loader = bids.layout.BIDSLayout(root=BIDS_DIR, validate=False, database_path=db_file)

bids_loader.add_derivatives(DIR_PIPELINES['preproc'])
bids_loader.add_derivatives(DIR_PIPELINES['uslr-lin'])

subjects = bids_loader.get_subjects()
print(f'Subjects : {subjects}')

## 4  Explore the tutorial subject

Subject **ADNI002S1155** has 13 T1w sessions spanning ~150 months.

In [ ]:
SUBJECT = subjects[0]
sessions = bids_loader.get_session(subject=SUBJECT)
print(f'Subject  : {SUBJECT}')
print(f'Sessions ({len(sessions)}): {sessions}')

# Show the sessions table
import pandas as pd
tsv_files = bids_loader.get(subject=SUBJECT, suffix='sessions', extension='tsv')
if tsv_files:
    df_sessions = pd.read_csv(tsv_files[0].path, sep='\t')
    display(df_sessions[['session_id', 'age', 'time_to_bl_months', 'dx', 'mmse']].drop_duplicates('session_id'))

In [ ]:
# Visual timeline of sessions
if tsv_files:
    df_plot = df_sessions.drop_duplicates('session_id').copy()
    df_plot = df_plot.sort_values('time_to_bl_months')

    fig, ax = plt.subplots(figsize=(12, 2))
    ax.scatter(df_plot['time_to_bl_months'], [0] * len(df_plot), s=80, zorder=3)
    ax.hlines(0, df_plot['time_to_bl_months'].min(), df_plot['time_to_bl_months'].max(),
              colors='gray', linewidths=1)
    for _, row in df_plot.iterrows():
        ax.text(row['time_to_bl_months'], 0.02, row['session_id'], ha='center', fontsize=8, rotation=45)
    ax.set_xlabel('Months from baseline')
    ax.set_yticks([])
    ax.set_title(f'Session timeline — sub-{SUBJECT}')
    plt.tight_layout()
    plt.show()

## 5  Prerequisite check

The linear registration requires SynthSeg segmentations (`*T1wdseg.nii.gz`) and bias-corrected images (`*T1w.nii.gz`) in `derivatives/preproc/`.  
Run this cell to see which sessions are ready.

In [ ]:
seg_entities = {'scope': 'preproc', 'extension': 'nii.gz', 'suffix': ['T1wdseg', 'dseg']}
bf_entities  = {'scope': 'preproc', 'extension': 'nii.gz', 'suffix': 'T1w', 'acquisition': [None, 'orig']}

print(f'Preproc dir : {DIR_PIPELINES["preproc"]}')
print()

rows = []
for ses in sessions:
    seg = bids_loader.get(subject=SUBJECT, session=ses, **seg_entities)
    bf  = bids_loader.get(subject=SUBJECT, session=ses, **bf_entities)
    rows.append({'session': ses, 'segmentation': '✓' if seg else '✗', 'bias-corrected T1w': '✓' if bf else '✗'})

df_preproc = pd.DataFrame(rows).set_index('session')
ready = (df_preproc == '✓').all(axis=1).sum()
print(f'{ready}/{len(sessions)} sessions have all preprocessing outputs.')
display(df_preproc)

if ready == 0:
    print('\n⚠️  No preprocessed sessions found.')
    print('    Run scripts/synthseg_segmentation.py and scripts/bias_field_correction.py first,')
    print('    or copy pre-computed derivatives into', DIR_PIPELINES['preproc'])

## 6  Initialise `USLR_Linear`

In [ ]:
processing = USLR_Linear(bids_loader=bids_loader, subject_list=[SUBJECT])

# Sessions that have segmentation outputs (used by the pipeline)
timepoints = processing._get_timepoints(subject=SUBJECT, uslr=True)
print(f'Sessions with preproc outputs ({len(timepoints)}): {timepoints}')

## 7  Run the linear registration

Equivalent to:
```bash
python scripts/linear_registration.py --bids <BIDS_DIR> --subjects ADNI002S1155
```

The pipeline is **resumable** — completed stages are detected and skipped automatically.  
Pass `force_flag=True` to rerun from scratch.

**Note:** `register_MNI=False` here — MNI registration is optional and slower.

In [ ]:
processing.process(force_flag=False, register_MNI=False)

## 8  Inspect outputs

### 8.1  Per-session affine matrices

Each matrix maps **raw session space → subject template space**.

In [ ]:
aff_graph_entities = {'desc': 'raw2temp', 'suffix': 'aff', 'extension': '.npy'}

affines = {}
for tp in timepoints:
    fname = processing.build_path({'session': tp, 'subject': SUBJECT, **aff_graph_entities})
    fpath = join(DIR_PIPELINES['uslr-lin'], fname)
    if exists(fpath):
        affines[tp] = np.load(fpath)

print(f'Loaded {len(affines)}/{len(timepoints)} affine matrices.')
if affines:
    tp0 = list(affines.keys())[0]
    print(f'\nAffine for session {tp0}:')
    print(np.round(affines[tp0], 4))

In [ ]:
if affines:
    n = len(affines)
    ncols = min(n, 4)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()

    for ax, (tp, M) in zip(axes, affines.items()):
        im = ax.imshow(M, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
        ax.set_title(f'ses-{tp}', fontsize=9)
        ax.set_xticks(range(4))
        ax.set_yticks(range(4))
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if abs(M[i, j]) > 0.5 else 'black')

    for ax in axes[len(affines):]:
        ax.set_visible(False)

    fig.colorbar(im, ax=axes[:len(affines)], shrink=0.6, label='Matrix value')
    fig.suptitle(f'Per-session affines (raw → template space)  —  sub-{SUBJECT}', fontsize=11)
    plt.tight_layout()
    plt.show()

### 8.2  Translation magnitude over time

How much does each session need to be shifted (in mm) relative to the template?

In [ ]:
if affines and tsv_files:
    df_time = df_sessions.drop_duplicates('session_id').set_index('session_id')

    times, translations, rotations = [], [], []
    for tp, M in affines.items():
        if tp in df_time.index:
            t_months = float(df_time.loc[tp, 'time_to_bl_months'])
            t_mm = np.linalg.norm(M[:3, 3])
            # rotation angle from the rotation matrix part
            R = M[:3, :3]
            cos_val = np.clip((np.trace(R) - 1) / 2, -1, 1)
            angle_deg = np.degrees(np.arccos(cos_val))
            times.append(t_months)
            translations.append(t_mm)
            rotations.append(angle_deg)

    idx = np.argsort(times)
    times = np.array(times)[idx]
    translations = np.array(translations)[idx]
    rotations = np.array(rotations)[idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(times, translations, 'o-', color='steelblue')
    ax1.set_xlabel('Months from baseline')
    ax1.set_ylabel('Translation magnitude (mm)')
    ax1.set_title('Head position shift relative to template')
    ax1.grid(True, alpha=0.3)

    ax2.plot(times, rotations, 'o-', color='tomato')
    ax2.set_xlabel('Months from baseline')
    ax2.set_ylabel('Rotation angle (°)')
    ax2.set_title('Head orientation relative to template')
    ax2.grid(True, alpha=0.3)

    plt.suptitle(f'Rigid registration parameters over time  —  sub-{SUBJECT}', fontsize=11)
    plt.tight_layout()
    plt.show()

### 8.3  Subject template image

Voxel-wise median of all registered sessions in the 192³ subject space.

In [ ]:
template_lin_entities = {'space': 'uslr', 'acquisition': '1', 'extension': '.nii.gz', 'suffix': 'T1w'}

fname_template = processing.build_path({'subject': SUBJECT, **template_lin_entities})
fpath_template = join(DIR_PIPELINES['uslr-lin'], fname_template)

fname_seg = processing.build_path({'subject': SUBJECT, **template_lin_entities, 'suffix': 'T1wdseg'})
fpath_seg = join(DIR_PIPELINES['uslr-lin'], fname_seg)

print(f'Template image : {fpath_template}')
print(f'Template seg   : {fpath_seg}')
print(f'Template exists: {exists(fpath_template)}')

In [ ]:
if exists(fpath_template):
    proxy = nib.load(fpath_template)
    vol = np.asarray(proxy.dataobj)
    print(f'Template shape : {vol.shape}')
    print(f'Voxel size     : {np.round(np.linalg.norm(proxy.affine[:3, :3], axis=0), 3)} mm')

    cx, cy, cz = [s // 2 for s in vol.shape]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    views = [
        (vol[cx, :, :],   f'Sagittal (x={cx})'),
        (vol[:, cy, :],   f'Coronal  (y={cy})'),
        (vol[:, :, cz],   f'Axial    (z={cz})'),
    ]
    for ax, (plane, title) in zip(axes, views):
        ax.imshow(np.rot90(plane), cmap='gray', interpolation='nearest')
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    fig.suptitle(f'Subject template (median T1w)  —  sub-{SUBJECT}', fontsize=12)
    plt.tight_layout()
    plt.show()

### 8.4  Template segmentation — label overlay

In [ ]:
if exists(fpath_template) and exists(fpath_seg):
    proxy_seg = nib.load(fpath_seg)
    seg = np.asarray(proxy_seg.dataobj)

    cx, cy, cz = [s // 2 for s in vol.shape]

    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    views = [
        (vol[cx, :, :],   seg[cx, :, :],   f'Sagittal'),
        (vol[:, cy, :],   seg[:, cy, :],   f'Coronal'),
        (vol[:, :, cz],   seg[:, :, cz],   f'Axial'),
    ]
    for col, (plane_im, plane_seg, title) in enumerate(views):
        axes[0, col].imshow(np.rot90(plane_im), cmap='gray', interpolation='nearest')
        axes[0, col].set_title(f'{title} — T1w', fontsize=10)
        axes[0, col].axis('off')

        axes[1, col].imshow(np.rot90(plane_im), cmap='gray', interpolation='nearest', alpha=0.6)
        axes[1, col].imshow(np.rot90(plane_seg.astype(float)),
                            cmap='nipy_spectral', alpha=0.5, interpolation='nearest',
                            vmin=0, vmax=seg.max())
        axes[1, col].set_title(f'{title} — seg overlay', fontsize=10)
        axes[1, col].axis('off')

    fig.suptitle(f'Template image + SynthSeg parcellation  —  sub-{SUBJECT}', fontsize=12)
    plt.tight_layout()
    plt.show()

### 8.5  eTIV (estimated total intracranial volume)

In [ ]:
sid = 'sub-' + SUBJECT
etiv_path = join(DIR_PIPELINES['uslr-lin'], sid, sid + '_T1wetiv.npy')

if exists(etiv_path):
    etiv_vox = float(np.load(etiv_path))
    # voxel volume in mm³ from the template affine
    if exists(fpath_template):
        vox_vol = np.abs(np.linalg.det(nib.load(fpath_template).affine))
        etiv_mm3 = etiv_vox * vox_vol
        print(f'eTIV : {etiv_vox:.0f} voxels  →  {etiv_mm3 / 1e3:.1f} cm³  ({etiv_mm3:.0f} mm³)')
    else:
        print(f'eTIV : {etiv_vox:.0f} voxels')
else:
    print(f'eTIV file not found at {etiv_path}')

### 8.6  Per-session registered images (sanity check)

Axial slice of each session after registration to the subject template space — all should look well-aligned.

In [ ]:
im_graph_lin_entities = {'space': 'uslr', 'acquisition': '1', 'extension': 'nii.gz', 'suffix': 'T1w'}

registered_files = []
for tp in timepoints:
    f = processing._get_data(**{'subject': SUBJECT, 'session': tp, **im_graph_lin_entities})
    if f is not None:
        registered_files.append((tp, f.path))

print(f'Found {len(registered_files)}/{len(timepoints)} registered session images.')

if registered_files:
    ncols = min(len(registered_files), 5)
    nrows = int(np.ceil(len(registered_files) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()

    for ax, (tp, fpath) in zip(axes, registered_files):
        v = np.asarray(nib.load(fpath).dataobj)
        z = v.shape[2] // 2
        ax.imshow(np.rot90(v[:, :, z]), cmap='gray', interpolation='nearest')
        ax.set_title(f'ses-{tp}', fontsize=8)
        ax.axis('off')

    for ax in axes[len(registered_files):]:
        ax.set_visible(False)

    fig.suptitle(f'Registered sessions (axial mid-slice, subject space)  —  sub-{SUBJECT}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 9  (Optional) Register to MNI

Uncomment and run the cell below to also map every session to MNI152 space.  
This calls `_register_to_MNI` with `model='linear'` (centroid-based affine only).

In [ ]:
# processing.process(force_flag=False, register_MNI=True)

### Check MNI outputs

In [ ]:
mni_files = bids_loader.get(subject=SUBJECT, space='MNI', suffix='T1w', extension='nii.gz')
print(f'MNI-space T1w files: {len(mni_files)}')
for f in mni_files:
    print(' ', f.path)

## 10  Output directory structure

Summary of what was written under `derivatives/uslr-lin/`.

In [ ]:
import subprocess
result = subprocess.run(['find', DIR_PIPELINES['uslr-lin'], '-name', '*.nii.gz', '-o', '-name', '*.npy'],
                        capture_output=True, text=True)
lines = sorted(result.stdout.strip().split('\n'))
print(f'Files in {DIR_PIPELINES["uslr-lin"]}:')
for l in lines:
    if l:
        print(' ', l.replace(DIR_PIPELINES['uslr-lin'] + '/', ''))